In [ ]:
%use lets-plot

import application.tester.TradingAlgorithmBackTester
import data.repository.historical_data.HistoricalMarketDataProvider
import domain.algorithm.TradingAlgorithm
import domain.market.security.SecurityIdentifier
import domain.tax.Taxation
import domain.trader.TradingOrder
import kotlinx.coroutines.async
import kotlinx.coroutines.awaitAll
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.runBlocking
import kotlin.time.Instant

enum class TradingOrderColor {
    BUY,
    SELL,
    BOTH,
    NOTHING,
}

//===========================================================//
//===========================================================//
// Config

val algorithm = TradingAlgorithm.Type.TACPP46
val taxation = Taxation.Type.Hungary
val startCapital = 1000.0
val startDate = Instant.parse("2020-01-01T00:00:00Z")
val endDate = Instant.parse("2026-01-01T00:00:00Z")

//===========================================================//

val listOfOutput = runBlocking {
    coroutineScope {
        HistoricalMarketDataProvider.getAllSecurityIdentifiers().getOrThrow().map {
            async {
                TradingAlgorithmBackTester(
                    type = algorithm,
                    securityIdentifier = it,
                    startingCapital = startCapital,
                    taxation = taxation,
                    from = startDate,
                    to = endDate
                ).runBackTest()
            }
        }.awaitAll()
    }
}

val plots = listOfOutput.map { output ->
    val days = output.stockHistory.mapIndexed { index, _ -> index }
    val stockPrice = output.stockHistory.map { it.closingPrice }
    val tradingOrders = output.tradingOrders.map { order ->
        if(order.buy == null && order.sell == null) TradingOrderColor.NOTHING
        else if(order.buy != null && order.sell != null) TradingOrderColor.BOTH
        else if(order.buy != null && order.sell == null) TradingOrderColor.BUY
        else TradingOrderColor.SELL
    }

    val plotData = mapOf(
        "day" to days,
        "stock_price" to stockPrice,
        "trading_orders" to tradingOrders
    )

    val pointIndices = tradingOrders.indices .filter { tradingOrders[it] != TradingOrderColor.NOTHING }
    val pointData = mapOf(
        "day" to pointIndices.map { days[it] },
        "stock_price" to pointIndices.map { stockPrice[it] },
        "trading_orders" to pointIndices.map { tradingOrders[it].name }
    )

    letsPlot(plotData) { x = "day"; y = "stock_price" } +
    geomLine(
        data = plotData,
        color = "steelblue",
        size = 1.2
    ) +
    geomPoint(
        data = pointData,
        size = 2.0
    ) {
        color = "trading_orders"
    } +
    scaleColorManual(
        values = mapOf(
            TradingOrderColor.BUY.name to "green",
            TradingOrderColor.SELL.name to "red",
            TradingOrderColor.BOTH.name to "orange"
        )
    ) +
    labs(
        title = "${output.tradingAlgorithmType} — ${output.securityIdentifier.tickerSymbol}",
        x = "Trading day",
        y = "Stock Price",
        color = "Trading Order"
    )
}

gggrid(
    plots,
    ncol = 4,
    hspace = 50,
    vspace = 100,
    align = true,
    fit = true
) +
ggsize(1800, 600) +
ggtb()

In [ ]:
%use lets-plot

import application.tester.TradingAlgorithmBackTester
import data.repository.historical_data.HistoricalMarketDataProvider
import domain.algorithm.TradingAlgorithm
import domain.market.security.SecurityIdentifier
import domain.tax.Taxation
import domain.trader.TradingOrder
import kotlinx.coroutines.async
import kotlinx.coroutines.awaitAll
import kotlinx.coroutines.coroutineScope
import kotlinx.coroutines.runBlocking
import kotlin.time.Instant

enum class TradingOrderColor {
    BUY,
    SELL,
    BOTH,
    NOTHING,
}

//===========================================================//
//===========================================================//
// Config

val algorithm = TradingAlgorithm.Type.TACPP462
val taxation = Taxation.Type.Hungary
val startCapital = 1000.0
val startDate = Instant.parse("2020-01-01T00:00:00Z")
val endDate = Instant.parse("2026-01-01T00:00:00Z")

val identifier = SecurityIdentifier(
    "US67066G1040", // NVDA
    //"US0378331005", // AAPL
    "NVDA",
    "USD"
)

//===========================================================//

val output = TradingAlgorithmBackTester(
    type = algorithm,
    securityIdentifier = identifier,
    startingCapital = startCapital,
    taxation = taxation,
    from = startDate,
    to = endDate
).runBackTest()

output.display()

val days = output.stockHistory.mapIndexed { index, _ -> index }
val stockPrice = output.stockHistory.map { it.closingPrice }
val tradingOrders = output.tradingOrders.map { order ->
    if(order.buy == null && order.sell == null) TradingOrderColor.NOTHING
    else if(order.buy != null && order.sell != null) TradingOrderColor.BOTH
    else if(order.buy != null && order.sell == null) TradingOrderColor.BUY
    else TradingOrderColor.SELL
}

val plotData = mapOf(
    "day" to days,
    "stock_price" to stockPrice,
    "trading_orders" to tradingOrders
)

val pointIndices = tradingOrders.indices .filter { tradingOrders[it] != TradingOrderColor.NOTHING }
val pointData = mapOf(
    "day" to pointIndices.map { days[it] },
    "stock_price" to pointIndices.map { stockPrice[it] },
    "trading_orders" to pointIndices.map { tradingOrders[it].name }
)

letsPlot(plotData) { x = "day"; y = "stock_price" } +
        geomLine(
            data = plotData,
            color = "steelblue",
            size = 1.2
        ) +
        geomPoint(
            data = pointData,
            size = 2.0
        ) {
            color = "trading_orders"
        } +
        scaleColorManual(
            values = mapOf(
                TradingOrderColor.BUY.name to "green",
                TradingOrderColor.SELL.name to "red",
                TradingOrderColor.BOTH.name to "orange"
            )
        ) +
        labs(
            title = "${output.tradingAlgorithmType} — ${output.securityIdentifier.tickerSymbol}",
            x = "Trading day",
            y = "Stock Price",
            color = "Trading Order"
        ) +
        ggsize(1800, 600) +
        ggtb()